In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 24. text text — text text text

> **Learning Goals**
> - Greedy, Beam Search, Top-k, Top-p, Temperaturetext text text
> - text text text text·text text text text
> - Speculative Decodingtext text text

## 24.1 text Problem text

LLMtext text text text text text Distribution $P(w_t | w_{<t})$ Output.
text text text text text = text.

text: **text(text) vs text(text)**

## 24.2 Greedy Decoding

text text text text text text text:
$$w_t = \arg\max_w P(w | w_{<t})$$

- text text
- text text, text text


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# text text text (10 text, text 20)
torch.manual_seed(0)
vocab_size = 20
n_steps = 10
logits_seq = torch.randn(n_steps, vocab_size)

def softmax(x, tau=1.0):
    x = x / tau
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

def greedy_decode(logits_seq):
    """Greedy: text Step argmax."""
    tokens = []
    for logits in logits_seq:
        tokens.append(logits.argmax().item())
    return tokens

greedy_tokens = greedy_decode(logits_seq)
print(f"Greedy: {greedy_tokens}")


## 24.3 Temperature Sampling

$$P_\tau(w) = \frac{\exp(\log P(w) / \tau)}{\sum_{w'} \exp(\log P(w') / \tau)}$$

- $\tau \to 0$: Greedytext text (text Distribution)
- $\tau = 1$: text text
- $\tau \to \infty$: text Distribution (text text)


In [ ]:
# Temperature text
logits = torch.randn(20)  # 20 text text text
print(f"text: {logits.numpy().round(2)}")
print()
for tau in [0.3, 1.0, 2.0, 5.0]:
    p = softmax(logits.numpy(), tau=tau)
    print(f"tau={tau}: {p.round(4)}  entropy={-np.sum(p*np.log(p+1e-12)):.3f}")
print("\n=> tau text text Distribution(text), text text(text).")


## 24.4 Top-k Sampling

text text text $k$text text text, text 0text:
$$\mathcal{V}_k = \text{top-}k\{P(w)\}$$
$$\hat{P}(w) = \frac{P(w)}{\sum_{w' \in \mathcal{V}_k} P(w')} \text{ for } w \in \mathcal{V}_k, \text{ else } 0$$

## 24.5 Top-p (Nucleus) Sampling

text text $p$ text text text text:
$$\mathcal{V}_p = \min \{S : \sum_{w \in S} P(w) \geq p\}$$

text: Distributiontext text text text, text text text.


In [ ]:
# Top-k, Top-p text
def top_k_sampling(logits, k=5, tau=1.0):
    """Top-k Sampletext."""
    p = softmax(logits.numpy(), tau=tau)
    # text ktext text
    top_idx = np.argsort(p)[-k:]
    mask = np.zeros_like(p)
    mask[top_idx] = p[top_idx]
    # Normalization
    mask = mask / mask.sum()
    return np.random.choice(len(p), p=mask)

def top_p_sampling(logits, p=0.9, tau=1.0):
    """Top-p (Nucleus) Sampletext."""
    probs = softmax(logits.numpy(), tau=tau)
    # text text text
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    # text
    cumsum = np.cumsum(sorted_probs)
    # p text text text text
    cutoff = np.searchsorted(cumsum, p) + 1
    # text cutoff text text
    keep = sorted_idx[:cutoff]
    new_p = np.zeros_like(probs)
    new_p[keep] = probs[keep]
    new_p = new_p / new_p.sum()
    return np.random.choice(len(probs), p=new_p)

# Visualization: text text text text text
np.random.seed(42)
logits = np.random.randn(20) * 2
probs = softmax(logits, tau=1.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Top-k text Calculation
top5_idx = np.argsort(probs)[-5:]
mask_topk = np.zeros(20)
mask_topk[top5_idx] = probs[top5_idx]
mask_topk = mask_topk / mask_topk.sum()

# Top-p text Calculation
sorted_idx = np.argsort(probs)[::-1]
sorted_probs = probs[sorted_idx]
cumsum = np.cumsum(sorted_probs)
cutoff = np.searchsorted(cumsum, 0.9) + 1
keep = sorted_idx[:cutoff]
mask_p = np.zeros(20); mask_p[keep] = probs[keep]
mask_p = mask_p / mask_p.sum()

# temperature
mask_t = softmax(logits, tau=0.5)

strategies = [
    ('Greedy', np.eye(20)[np.argmax(probs)]),
    ('Top-k=5', mask_topk),
    ('Top-p=0.9', mask_p),
    ('Temperature=0.5', mask_t),
]

for ax, (name, mask) in zip(axes, strategies):
    ax.bar(range(20), probs, alpha=0.3, color='gray', label='Original Distribution')
    ax.bar(range(20), mask, alpha=0.7, color='red', label='Linetext text')
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ch24_decoding_strategies.png', dpi=100, bbox_inches='tight')
plt.show()


## 24.6 Beam Search

$B$text text text(beam)text text, text text text:

text: $\sum_t \log P(w_t | w_{<t})$

- text text $B$ = text 4~10
- text text text text
- LLMtext text text text (text, text)


In [ ]:
# Beam Search text text
def beam_search(initial_tokens, get_logits_fn, beam_width=4, max_len=20):
    """text beam search."""
    beams = [(initial_tokens, 0.0)]  # (tokens, log_prob)
    for step in range(max_len):
        all_candidates = []
        for tokens, score in beams:
            logits = get_logits_fn(tokens)
            log_probs = F.log_softmax(logits[-1], dim=-1)
            # text beam_widthtext
            topk = log_probs.topk(beam_width)
            for i in range(beam_width):
                new_tokens = tokens + [topk.indices[i].item()]
                new_score = score + topk.values[i].item()
                all_candidates.append((new_tokens, new_score))
        # text beam_widthtext Linetext
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_width]
    return beams[0]  # text Scores

# text text Function
def fake_logits_fn(tokens):
    return torch.randn(len(tokens), 50)  # text 50

result_tokens, result_score = beam_search([1, 2, 3], fake_logits_fn, beam_width=4, max_len=10)
print(f"Beam Search Result: tokens={result_tokens}, score={result_score:.4f}")


## 24.7 Speculative Decoding

**text**: text text Model(draft)text text $k$text text text, text Model(target)text text Verification.

1. Draft Model: $w_1^d, \ldots, w_k^d$ text (text)
2. Target Model: $P_T(w_1), \ldots, P_T(w_k)$ text Calculation (text text Forward Pass)
3. text text, text text text text

text: 2-3x Speedup (Medusa, EAGLE text)


## 24.8 Repetition Penalty

text text: text text text text text.

$$\logit'(w) = \begin{cases} \logit(w) / p & \text{if } w \in \text{generated} \\ \logit(w) & \text{otherwise} \end{cases}$$

$p > 1$ (text 1.1~1.3).

## 24.9 Key Takeaways

| text | text | text |
|---|---|---|
| Greedy | $\arg\max P$ | text, text |
| Temperature | $\sigma(z/\tau)$ | Distribution text text |
| Top-k | $\text{top-}k$ text | text k |
| Top-p | text $p$ | text |
| Beam | $B$text text | text ↑, text ↓ |
| Speculative | draft + target | 2-3x text |

## Exercises

1. Greedy, Top-k=5, Top-p=0.9, Temperature=0.7text text text text Comparisontext.
2. Temperature 0.3, 0.7, 1.0, 1.5text text text text text Comparisontext.
3. Beam Searchtext text text 1, 4, 8text Comparisontext.
4. Repetition Penalty 1.0, 1.1, 1.3text text text.
5. Speculative Decodingtext 2-3x text text text text.

> Solutions: `solutions/ch24_solutions.ipynb`
